# Loading and preparing data 

In [ ]:
import scvi
import scanpy as sc

import pandas as pd
import subprocess
import os

In [ ]:
import anndata as ad

In [ ]:
print(scvi.__version__)

In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
scvi.settings.num_threads=48

# Import AnnData

In [9]:
adata = sc.read_h5ad('data/object/qc.h5ad')
adata

AnnData object with n_obs × n_vars = 69837 × 11874
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample_path', 'sample_dir', 'patient_name', 'time_point', 'digestion_protocol', 'patient_id', 'sample_name', 'hto_demux_class', 'pMT_RNA', 'pHB_RNA', 'pRP_RNA', 'cell_id', 'msS_RNA', 'msG2M_RNA', 'cc_phase_class', 'msCC_diff_RNA', 'celltypist_predicted_labels_low', 'celltypist_predicted_labels_high', 'solo_pred_doublet', 'solo_pred_singlet', 'solo_label', 'solo_mad_label', 'pMT_RNA_min', 'pMT_RNA_max', 'pHB_RNA_min', 'pHB_RNA_max', 'nCount_RNA_min', 'nCount_RNA_max', 'qc_class', 'nFeature_RNA_max', 'nFeature_RNA_min', 'RNA_snn_res.1', 'seurat_clusters'

# Register model

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata,
    categorical_covariate_keys=['patient_id', 'time_point', 'digestion_protocol'], 
    continuous_covariate_keys=['pMT_RNA', 'msCC_diff_RNA']
)

# Creating and training a model 

In [ ]:
model = scvi.model.SCVI(
    adata, 
    n_latent=10, 
    n_hidden=128, 
    n_layers=2, 
    gene_likelihood='nb'
)

In [ ]:
model.train(max_epochs=200)

In [ ]:
model.save('data/object/scvi/model/', overwrite=True)

# Update anndata object with scvi results 

In [ ]:
adata.layers['data'] = model.get_normalized_expression(library_size=10e4)
adata.obsm['latent'] = model.get_latent_representation()

# Dimensional reduction on scvi latent space 

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30, use_rep='latent')
sc.tl.leiden(adata, resolution=1, key_added='leiden')
sc.tl.umap(adata, min_dist=1)

## UMAP

In [ ]:
sc.pl.umap(adata, color=['leiden', 'patient_id', 'time_point', 'hto_demux_class', 'digestion_protocol', 'cc_phase_class', 'pMT_RNA', 'pHB_RNA', 'pRP_RNA', 'nCount_RNA', 'nFeature_RNA'], frameon=False, ncols=5, wspace=0.5)

# Save results

In [ ]:
adata.write('data/object/scvi/adata.h5ad')
adata.write('data/object/pp.h5ad')

In [ ]:
pd.DataFrame(adata.obsm['X_umap'], index=adata.obs.index).to_csv('data/object/components/umap.csv')
pd.DataFrame(adata.obs, index=adata.obs.index).to_csv('data/object/components/meta.csv')